# Phân Tích Cảm Xúc Đa Khía Cạnh (ABSA) - BiLSTM + Word2vec

## 1. Cài đặt thư viện


In [1]:
# Cài đặt pyvi để tách từ tiếng Việt và gensim để huấn luyện Word2Vec
!pip install -q pyvi gensim iterative-stratification seaborn scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 57.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.2 MB/s eta 0:00:00


In [2]:
pip install nlpaug

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.5/410.5 kB 7.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install -q transformers

## 2. Import thư viện


In [4]:
import json, math, os, random, warnings
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from pyvi import ViTokenizer
from gensim.models import Word2Vec

import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight

import nlpaug.augmenter.word as naw
import copy

try:
    from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
    HAS_ITERSTRAT = True
except Exception:
    HAS_ITERSTRAT = False

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


## 3. Cấu hình


In [5]:

# --- CẤU HÌNH BILSTM & WORD2VEC ---
MAX_LENGTH = 160
EMBEDDING_DIM = 300
HIDDEN_DIM = 256
NUM_LAYERS = 2
DROPOUT = 0.4

# --- CẤU HÌNH HUẤN LUYỆN ---
EPOCHS = 8
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
USE_ASPECT_ATTENTION = True

ABSENT_CLASS = 3
SENTIMENT_CLASSES = [0, 1, 2]
ASPECT_COLS = ['as_content', 'as_physical', 'as_price', 'as_packaging', 'as_delivery', 'as_service']
LABEL_COLS = ['sentiment_llm', *ASPECT_COLS]
ASPECT_DISPLAY = {col: col.split('_')[1].capitalize() for col in ASPECT_COLS}


DATA_ROOT = Path('/kaggle/input/datasets/jyang10/tiki-cleaned-book-reviews')
if not DATA_ROOT.exists():
    DATA_ROOT = Path('.') # Fallback
TRAIN_PATH = DATA_ROOT / 'train_clean.json'
VAL_PATH = DATA_ROOT / 'val_clean.json'
TEST_PATH = DATA_ROOT / 'test_clean.json'

OUTPUT_ROOT = Path('./absa_bilstm_word2vec')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## 4. Nạp và chuẩn bị dữ liệu


In [6]:
def load_and_prepare(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    df = pd.read_json(path)

    title = df.get('review_title', df.get('title', pd.Series(['']*len(df)))).fillna('').astype(str).str.strip()
    body = df.get('content', df.get('text', pd.Series(['']*len(df)))).fillna('').astype(str).str.strip()

    df['title'] = title
    df['body'] = body
    df['text_full'] = df['title'] + " " + df['body']

    df['sentiment_llm'] = pd.to_numeric(df['sentiment_llm'], errors='coerce')
    df = df.dropna(subset=['sentiment_llm'])
    df['sentiment_llm'] = df['sentiment_llm'].astype(int)
    df = df[df['sentiment_llm'].isin(SENTIMENT_CLASSES)]

    for col in ASPECT_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(ABSENT_CLASS).astype(int).clip(0, ABSENT_CLASS)

    return df[['text_full', *LABEL_COLS]].reset_index(drop=True)

train_df = load_and_prepare(TRAIN_PATH)
val_df = load_and_prepare(VAL_PATH)
test_df = load_and_prepare(TEST_PATH)

print(f"Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Train: (9360, 8), Val: (2009, 8), Test: (2005, 8)


## 6. Tách từ & Huấn luyện Word2Vec


In [7]:
from pyvi import ViTokenizer
from transformers import AutoTokenizer

print("Đang tách từ (Word Segmentation) bằng Pyvi...")

# Hàm tách từ tiếng Việt
def segment_text(text):
    return ViTokenizer.tokenize(text.lower())

# Khôi phục lại bước tạo cột 'text_seg' đã bị khuyết thiếu
train_df['text_seg'] = train_df['text_full'].apply(segment_text)
val_df['text_seg'] = val_df['text_full'].apply(segment_text)
test_df['text_seg'] = test_df['text_full'].apply(segment_text)

print("Đang nạp PhoBERT Tokenizer...")
# Sử dụng PhoBERT base
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# Cập nhật số chiều embedding của PhoBERT
EMBEDDING_DIM = 768

print("Đã chuẩn bị xong dữ liệu và Tokenizer!")

Đang tách từ (Word Segmentation) bằng Pyvi...
Đang nạp PhoBERT Tokenizer...


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Đã chuẩn bị xong dữ liệu và Tokenizer!


## 7. Dataset & DataLoader


In [8]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text_seg']) # Văn bản đã được tách từ bằng pyvi

        # Sử dụng tokenizer của HuggingFace
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        labels = np.array([int(row['sentiment_llm']), *[int(row[col]) for col in ASPECT_COLS]], dtype=np.int64)

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(labels, dtype=torch.long)
        }

# Khởi tạo lại Dataloader
train_dataset = TextDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = TextDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = TextDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True) # GIẢM BATCH_SIZE XUỐNG 32 hoặc 16 VÌ PHOBERT RẤT NẶNG
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 8. Kiến trúc mô hình


In [9]:
from transformers import AutoModel

class SpatialDropout1D(nn.Module):
    def __init__(self, p):
        super(SpatialDropout1D, self).__init__()
        self.dropout = nn.Dropout2d(p)
    def forward(self, x):
        x = x.permute(0, 2, 1).unsqueeze(3)
        x = self.dropout(x)
        x = x.squeeze(3).permute(0, 2, 1)
        return x
        
class PhoBERT_ABSABiLSTM(nn.Module):
    def __init__(self, hidden_dim, num_layers, dropout, num_aspects):
        super(PhoBERT_ABSABiLSTM, self).__init__()
        
        # 1. PhoBERT thay thế cho Word2Vec
        self.phobert = AutoModel.from_pretrained("vinai/phobert-base")
        embed_dim = self.phobert.config.hidden_size # 768
        
        self.spatial_dropout = SpatialDropout1D(dropout)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim, hidden_size=hidden_dim, num_layers=num_layers,
            batch_first=True, bidirectional=True, dropout=dropout if num_layers > 1 else 0
        )
        
        lstm_out_dim = hidden_dim * 2
        self.num_heads = 4
        self.mha = nn.MultiheadAttention(
            embed_dim=lstm_out_dim, num_heads=self.num_heads, dropout=dropout, batch_first=True
        )
        
        self.attention_pool = nn.Sequential(
            nn.Linear(lstm_out_dim, lstm_out_dim // 2), nn.Tanh(), nn.Linear(lstm_out_dim // 2, 1)
        )
        
        self.feat_norm = nn.LayerNorm(lstm_out_dim * 3)
        cat_dim = lstm_out_dim * 3 
        
        # --- SENTIMENT DECOUPLER ---
        self.sent_proj = nn.Linear(cat_dim, hidden_dim)
        self.sent_decoupler = nn.Sequential(
            nn.Linear(cat_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU()
        )
        self.sent_classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout / 2), nn.Linear(hidden_dim, 3)
        )
        
        # --- ASPECT DECOUPLER ---
        self.asp_proj = nn.Linear(cat_dim, hidden_dim)
        self.asp_decoupler = nn.Sequential(
            nn.Linear(cat_dim, hidden_dim), nn.LayerNorm(hidden_dim),
            nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU()
        )
        
        self.pres_classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout / 2), nn.Linear(hidden_dim, num_aspects * 2)
        )
        self.asp_classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Dropout(dropout / 2), nn.Linear(hidden_dim, num_aspects * 3)
        )
        self.num_aspects = num_aspects

    def forward(self, input_ids, attention_mask):
        # Trích xuất đặc trưng ngữ cảnh từ PhoBERT
        # Lấy hidden_state của lớp cuối cùng
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        embedded = outputs.last_hidden_state 
        
        embedded = self.spatial_dropout(embedded)
        
        # Tạo lengths động từ attention_mask cho pack_padded_sequence
        lengths = attention_mask.sum(dim=1).clamp(min=1).cpu()
        
        packed_embedded = nn.utils.rnn.pack_padded_sequence(embedded, lengths, batch_first=True, enforce_sorted=False)
        packed_output, _ = self.lstm(packed_embedded)
        lstm_out, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True)
        
        batch_size, max_len, _ = lstm_out.size()
        # Tạo mask cho MHA và Pooling
        mask = torch.arange(max_len).expand(batch_size, max_len).to(lstm_out.device) < lengths.unsqueeze(1).to(lstm_out.device)
        key_padding_mask = ~mask
        
        attn_output, _ = self.mha(query=lstm_out, key=lstm_out, value=lstm_out, key_padding_mask=key_padding_mask)
        mask_expanded = mask.unsqueeze(-1)
        
        attn_weights = self.attention_pool(attn_output)
        attn_weights = attn_weights.masked_fill(~mask_expanded, -1e9)
        attn_weights = torch.softmax(attn_weights, dim=1)
        mhsa_pool = torch.sum(attn_weights * attn_output, dim=1)
        
        max_pool = torch.max(lstm_out.masked_fill(~mask_expanded, -1e9), dim=1)[0]
        
        mask_float = mask.float().unsqueeze(-1)
        avg_pool = torch.sum(lstm_out * mask_float, dim=1) / torch.sum(mask_float, dim=1).clamp(min=1e-9)
        
        context_vector = torch.cat([mhsa_pool, max_pool, avg_pool], dim=-1)
        context_vector = self.feat_norm(context_vector)
        
        residual_sent = self.sent_proj(context_vector)
        isolated_sent_context = self.sent_decoupler(context_vector) + residual_sent
        sent_logits = self.sent_classifier(isolated_sent_context)
        
        residual_asp = self.asp_proj(context_vector)
        isolated_asp_context = self.asp_decoupler(context_vector) + residual_asp
            
        pres_logits = self.pres_classifier(isolated_asp_context).view(-1, self.num_aspects, 2)
        asp_logits = self.asp_classifier(isolated_asp_context).view(-1, self.num_aspects, 3)
        
        return sent_logits, pres_logits, asp_logits

model = PhoBERT_ABSABiLSTM(
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout=DROPOUT, num_aspects=len(ASPECT_COLS)
).to(DEVICE)


    
if torch.cuda.device_count() > 1:
    print(f" Kích hoạt Multi-GPU: Sử dụng {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)
print("Mô hình đã được khởi tạo thành công!")

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
lm_head.decoder.weight          | UNEXPECTED |  | 
lm_head.decoder.bias            | UNEXPECTED |  | 
lm_head.dense.bias              | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
lm_head.layer_norm.weight       | UNEXPECTED |  | 
lm_head.bias                    | UNEXPECTED |  | 
lm_head.layer_norm.bias         | UNEXPECTED |  | 
lm_head.dense.weight            | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

 Kích hoạt Multi-GPU: Sử dụng 2 GPUs!
Mô hình đã được khởi tạo thành công!


## 9. Loss, metrics và trainer


In [10]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.weight = weight 
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean': return focal_loss.mean()
        elif self.reduction == 'sum': return focal_loss.sum()
        return focal_loss

# --- MỚI: TỰ ĐỘNG CÂN BẰNG MULTI-TASK LOSS BẰNG UNCERTAINTY ---
class AutomaticWeightedLoss(nn.Module):
    def __init__(self, num=3):
        super(AutomaticWeightedLoss, self).__init__()
        # Khởi tạo 3 tham số học được (cho Sent, Pres, Asp)
        self.params = nn.Parameter(torch.zeros(num, requires_grad=True))

    def forward(self, loss_sent, loss_pres, loss_asp):
        # Công thức tính theo Uncertainty Weighting (Kendall et al.)
        loss0 = loss_sent * torch.exp(-self.params[0]) + self.params[0]
        loss1 = loss_pres * torch.exp(-self.params[1]) + self.params[1]
        loss2 = loss_asp * torch.exp(-self.params[2]) + self.params[2]
        return loss0 + loss1 + loss2

def get_class_weights(labels_array, num_classes=3):
    weights = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=labels_array)
    smoothed_weights = np.sqrt(weights) 
    return torch.tensor(smoothed_weights, dtype=torch.float32)

# Hàm này giờ trả về 3 loss riêng lẻ, việc cộng gộp do AutomaticWeightedLoss lo
def compute_individual_losses(sent_logits, pres_logits, asp_logits, labels, sent_weights, pres_weights):
    true_sent = labels[:, 0]
    true_aspects = labels[:, 1:] 
    
    # 1. Loss Sentiment (Dùng Focal Loss)
    criterion_sent = FocalLoss(weight=sent_weights, gamma=2.0)
    loss_sent = criterion_sent(sent_logits, true_sent)
    
    # 2. Loss Presence
    true_pres = (true_aspects != ABSENT_CLASS).long()
    loss_pres = 0.0
    num_aspects = true_aspects.shape[1]
    
    for i in range(num_aspects):
        logits_i = pres_logits[:, i, :] 
        true_i = true_pres[:, i]        
        criterion_pres_i = nn.CrossEntropyLoss(weight=pres_weights[i])
        loss_pres += criterion_pres_i(logits_i, true_i)
    loss_pres = loss_pres / num_aspects 
    
    # 3. Loss Aspect Sentiment
    criterion_asp = nn.CrossEntropyLoss(label_smoothing=0.05)
    mask = true_aspects != ABSENT_CLASS 
    
    if mask.sum() > 0:
        asp_logits_flat = asp_logits.reshape(-1, 3)
        true_aspects_flat = true_aspects.reshape(-1)
        mask_flat = mask.reshape(-1)
        loss_asp = criterion_asp(asp_logits_flat[mask_flat], true_aspects_flat[mask_flat])
    else:
        loss_asp = torch.tensor(0.0, device=sent_logits.device, requires_grad=True)
        
    return loss_sent, loss_pres, loss_asp

def calculate_metrics(all_labels, all_preds):
    true_sent, true_asp = all_labels[:, 0], all_labels[:, 1:]
    pred_sent, pred_asp = all_preds[:, 0], all_preds[:, 1:]
    
    f1_sent = f1_score(true_sent, pred_sent, average='macro', zero_division=0)
    mask = true_asp != ABSENT_CLASS
    f1_asp_present = f1_score(true_asp[mask], pred_asp[mask], average='macro', zero_division=0) if mask.any() else 0.0
    f1_asp_all = f1_score(true_asp.flatten(), pred_asp.flatten(), labels=[0,1,2,3], average='macro', zero_division=0)
    
    f1_final = 0.5 * f1_sent + 0.5 * f1_asp_present
    return f1_sent, f1_asp_present, f1_asp_all, f1_final

## 10.Training Loop


In [11]:
awl = AutomaticWeightedLoss(num=3).to(DEVICE)

# Chia Parameter Groups: PhoBERT học siêu chậm, BiLSTM học nhanh
phobert_params = list(model.module.phobert.parameters()) if hasattr(model, 'module') else list(model.phobert.parameters())
custom_params = [p for n, p in model.named_parameters() if 'phobert' not in n]

optimizer = torch.optim.AdamW([
    {'params': phobert_params, 'lr': 2e-5},           
    {'params': custom_params, 'lr': LEARNING_RATE},   
    {'params': awl.parameters(), 'weight_decay': 0.0, 'lr': LEARNING_RATE}
], weight_decay=1e-4)

scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
best_val_f1 = 0.0

print("Đang tính toán Class Weights động cho Overall Sentiment...")
train_sent_labels = train_df['sentiment_llm'].values
dynamic_sent_weights = get_class_weights(train_sent_labels).to(DEVICE)
print(f"Trọng số Sentiment đã tính toán: {dynamic_sent_weights.cpu().numpy()}")

print("Đang tính toán Class Weights động cho Aspect Presence...")
aspect_pres_weights = []
for col in ASPECT_COLS:
    presence_labels = (train_df[col].values != ABSENT_CLASS).astype(int) 
    weights = compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=presence_labels)
    smoothed_weights = np.sqrt(weights)
    aspect_pres_weights.append(smoothed_weights)

aspect_pres_weights = torch.tensor(np.array(aspect_pres_weights), dtype=torch.float32).to(DEVICE)
print(f"Trọng số Aspect Presence đã tính: \n{aspect_pres_weights.cpu().numpy()}")

print("=" * 70)
print("Bắt đầu huấn luyện với PhoBERT + Aspect Decoupler + Uncertainty MTL...")
print("=" * 70)

for epoch in range(EPOCHS):
    # ==========================
    # 1. TRAINING PHASE
    # ==========================
    # ĐÂY LÀ PHẦN BỊ MẤT TRƯỚC ĐÓ - Đã được khôi phục
    model.train()
    awl.train() 
    total_train_loss = 0
    train_labels, train_preds = [], []

    for batch in train_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE) 
        labels = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        sent_logits, pres_logits, asp_logits = model(input_ids, attention_mask)

        loss_sent, loss_pres, loss_asp = compute_individual_losses(
            sent_logits, pres_logits, asp_logits, labels, dynamic_sent_weights, aspect_pres_weights
        )
        loss = awl(loss_sent, loss_pres, loss_asp)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()

        with torch.no_grad():
            pred_sent = sent_logits.argmax(dim=-1).cpu().numpy()
            pred_pres = pres_logits.argmax(dim=-1).cpu().numpy()
            pred_asp_sent = asp_logits.argmax(dim=-1).cpu().numpy()

            pred_asp = np.where(pred_pres == 0, ABSENT_CLASS, pred_asp_sent)
            preds = np.column_stack((pred_sent, pred_asp))

            train_labels.extend(labels.cpu().numpy())
            train_preds.extend(preds)

    # ==========================
    # 2. VALIDATION PHASE
    # ==========================
    model.eval()
    awl.eval()
    total_val_loss = 0
    val_labels, val_preds = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE) # Sửa lỗi KeyError tiềm ẩn
            labels = batch['labels'].to(DEVICE)

            sent_logits, pres_logits, asp_logits = model(input_ids, attention_mask)

            loss_sent, loss_pres, loss_asp = compute_individual_losses(
                sent_logits, pres_logits, asp_logits, labels, dynamic_sent_weights, aspect_pres_weights
            )
            val_loss = awl(loss_sent, loss_pres, loss_asp)
            total_val_loss += val_loss.item()

            pred_sent = sent_logits.argmax(dim=-1).cpu().numpy()
            pred_pres = pres_logits.argmax(dim=-1).cpu().numpy()
            pred_asp_sent = asp_logits.argmax(dim=-1).cpu().numpy()

            pred_asp = np.where(pred_pres == 0, ABSENT_CLASS, pred_asp_sent)
            preds = np.column_stack((pred_sent, pred_asp))

            val_labels.extend(labels.cpu().numpy()) 
            val_preds.extend(preds)

    # ==========================
    # 3. CALCULATE METRICS & LOG
    # ==========================
    train_f1_sent, train_f1_asp_pres, _, train_f1_final = calculate_metrics(np.array(train_labels), np.array(train_preds))
    val_f1_sent, val_f1_asp_pres, _, val_f1_final = calculate_metrics(np.array(val_labels), np.array(val_preds))

    avg_train_loss = total_train_loss / len(train_loader)
    avg_val_loss = total_val_loss / len(val_loader)
    current_lr_phobert = optimizer.param_groups[0]['lr']
    current_lr_bilstm = optimizer.param_groups[1]['lr']

    print(f"Epoch {epoch+1}/{EPOCHS} | LR (PhoBERT): {current_lr_phobert:.1e} | LR (BiLSTM): {current_lr_bilstm:.1e}")
    learned_weights = torch.exp(-awl.params).detach().cpu().numpy()
    print(f"  [Task Weights] Sent: {learned_weights[0]:.3f} | Pres: {learned_weights[1]:.3f} | Asp: {learned_weights[2]:.3f}")
    
    print(f"  [Train] Loss: {avg_train_loss:.4f} | F1 Final: {train_f1_final:.4f} (Sent: {train_f1_sent:.4f}, Asp: {train_f1_asp_pres:.4f})")
    print(f"  [Val]   Loss: {avg_val_loss:.4f} | F1 Final: {val_f1_final:.4f} (Sent: {val_f1_sent:.4f}, Asp: {val_f1_asp_pres:.4f})")

    scheduler.step(val_f1_final)

    if val_f1_final > best_val_f1:
        best_val_f1 = val_f1_final
        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save(model_to_save.state_dict(), OUTPUT_ROOT / 'best_bilstm_model.pt')
        print("  -->  Đã lưu mô hình tốt nhất!")

    print("-" * 70)

Đang tính toán Class Weights động cho Overall Sentiment...
Trọng số Sentiment đã tính toán: [0.7979566 1.4355369 1.0291079]
Đang tính toán Class Weights động cho Aspect Presence...
Trọng số Aspect Presence đã tính: 
[[0.8553989  1.2565618 ]
 [0.90627855 1.1304823 ]
 [0.7420163  2.3327796 ]
 [0.7808281  1.6670625 ]
 [0.8193612  1.3996352 ]
 [0.71879786 3.9365833 ]]
Bắt đầu huấn luyện với PhoBERT + Aspect Decoupler + Uncertainty MTL...
Epoch 1/8 | LR (PhoBERT): 2.0e-05 | LR (BiLSTM): 1.0e-03
  [Task Weights] Sent: 1.260 | Pres: 1.378 | Asp: 1.262
  [Train] Loss: 1.1750 | F1 Final: 0.5543 (Sent: 0.7226, Asp: 0.3861)
  [Val]   Loss: 0.5602 | F1 Final: 0.6561 (Sent: 0.8075, Asp: 0.5047)
  -->  Đã lưu mô hình tốt nhất!
----------------------------------------------------------------------
Epoch 2/8 | LR (PhoBERT): 2.0e-05 | LR (BiLSTM): 1.0e-03
  [Task Weights] Sent: 1.676 | Pres: 1.882 | Asp: 1.589
  [Train] Loss: 0.1691 | F1 Final: 0.6622 (Sent: 0.8135, Asp: 0.5109)
  [Val]   Loss: -0.0940

## 11. Ma trận nhầm lẫn


In [13]:
# Định nghĩa tên khía cạnh tiếng Việt để hiển thị cho đẹp
ASPECT_NAMES = ['Nội dung', 'Hình thức', 'Giá cả', 'Đóng gói', 'Giao hàng', 'Dịch vụ']

def plot_confusion_matrices(true_sentiment: np.ndarray, pred_sentiment: np.ndarray, true_aspects: np.ndarray, pred_aspects: np.ndarray) -> None:
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.flatten()

    # Nhãn cảm xúc: 0 (tiêu cực), 1 (trung tính), 2 (tích cực)
    sent_cm = confusion_matrix(true_sentiment, pred_sentiment, labels=[0, 1, 2])
    sns.heatmap(sent_cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'], ax=axes[0])
    axes[0].set_title('Overall sentiment')

    for idx, col in enumerate(ASPECT_COLS):
        ax = axes[idx + 1]
        # Bỏ qua các khía cạnh không xuất hiện (ABSENT_CLASS = 3) trong nhãn thực tế
        mask = true_aspects[:, idx] != ABSENT_CLASS
        if mask.sum() == 0:
            ax.set_visible(False)
            continue

        cm = confusion_matrix(true_aspects[:, idx][mask], pred_aspects[:, idx][mask], labels=[0, 1, 2])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges', cbar=False,
                    xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'], ax=ax)
        ax.set_title(f'{ASPECT_NAMES[idx]} (n={mask.sum()})')

    # Ẩn ô cuối cùng nếu dư
    if len(axes) > 7:
        axes[7].set_visible(False)

    plt.suptitle('BiLSTM + Word2Vec ABSA - Test confusion matrices', y=1.01, fontweight='bold', fontsize=16)
    plt.tight_layout()

    # Lưu ảnh ma trận nhầm lẫn
    os.makedirs(OUTPUT_ROOT, exist_ok=True)
    plt.savefig(OUTPUT_ROOT / 'confusion_matrix_test.png', dpi=150, bbox_inches='tight')
    plt.show()

# ==========================================
# THỰC THI ĐÁNH GIÁ TRÊN TẬP TEST
# ==========================================
print("Đang đánh giá mô hình BiLSTM trên tập Test...")

# Load mô hình tốt nhất một cách an toàn (tương thích cả Single-GPU và Multi-GPU)
state_dict = torch.load(OUTPUT_ROOT / 'best_bilstm_model.pt', map_location=DEVICE)

if hasattr(model, 'module'):
    model.module.load_state_dict(state_dict)
else:
    model.load_state_dict(state_dict)

model.eval()

true_sentiment, pred_sentiment = [], []
true_aspects, pred_aspects = [], []

# Chạy suy luận (Inference) trên Test DataLoader
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE) # SỬA LẠI Ở ĐÂY
        labels = batch['labels'].numpy()

        # Lấy Logits từ mô hình, truyền attention_mask vào
        sent_logits, pres_logits, asp_logits = model(input_ids, attention_mask)

        # Lấy nhãn dự đoán bằng argmax
        p_sent = sent_logits.argmax(dim=-1).cpu().numpy()
        p_pres = pres_logits.argmax(dim=-1).cpu().numpy()
        p_asp_sent = asp_logits.argmax(dim=-1).cpu().numpy()

        # Logic ghép nhãn: Nếu presence = 0 (không xuất hiện) -> gán nhãn ABSENT_CLASS (3)
        p_asp = np.where(p_pres == 0, ABSENT_CLASS, p_asp_sent)

        # Lưu trữ kết quả
        true_sentiment.extend(labels[:, 0])
        pred_sentiment.extend(p_sent)
        true_aspects.extend(labels[:, 1:])
        pred_aspects.extend(p_asp)

# Chuyển đổi sang Numpy Array
true_sentiment = np.array(true_sentiment)
pred_sentiment = np.array(pred_sentiment)
true_aspects = np.array(true_aspects)
pred_aspects = np.array(pred_aspects)

# 3. In Báo cáo phân loại (Classification Report)
print('\n=========================================')
print('=== BÁO CÁO TỔNG THỂ (OVERALL SENTIMENT) ===')
print('=========================================')
print(classification_report(
    true_sentiment,
    pred_sentiment,
    labels=[0, 1, 2],
    target_names=['neg', 'neu', 'pos'],
    zero_division=0
))

# ==========================================
# 5. BÁO CÁO TRUNG BÌNH TOÀN BỘ ASPECT (present only)
# ==========================================

all_true = []
all_pred = []

for idx in range(len(ASPECT_COLS)):
    t_asp = true_aspects[:, idx]
    p_asp = pred_aspects[:, idx]

    mask = t_asp != ABSENT_CLASS  # chỉ lấy các aspect có xuất hiện

    if mask.sum() == 0:
        continue

    all_true.extend(t_asp[mask])
    all_pred.extend(p_asp[mask])

all_true = np.array(all_true)
all_pred = np.array(all_pred)

print('\n=========================================')
print('=== Báo Cáo Mean 6 ASPECTS')
print('=========================================')

print(classification_report(
    all_true,
    all_pred,
    labels=[0, 1, 2],
    target_names=['neg', 'neu', 'pos'],
    zero_division=0
))



print('\n=========================================')
print('=== BÁO CÁO CHI TIẾT TỪNG KHÍA CẠNH ===')
print('=========================================')
for idx, (col_name, aspect_name) in enumerate(zip(ASPECT_COLS, ASPECT_NAMES)):
    # Lấy nhãn thực tế và nhãn dự đoán cho khía cạnh hiện tại
    t_asp = true_aspects[:, idx]
    p_asp = pred_aspects[:, idx]

    # Tạo mask để lọc bỏ những mẫu không chứa khía cạnh này (ABSENT_CLASS)
    mask = t_asp != ABSENT_CLASS

    print(f'\n--- Khía cạnh: {aspect_name.upper()} (n = {mask.sum()}) ---')

    if mask.sum() > 0:
        print(classification_report(
            t_asp[mask],
            p_asp[mask],
            labels=[0, 1, 2],
            target_names=['neg', 'neu', 'pos'],
            zero_division=0
        ))
    else:
        print(f"Không có dữ liệu thực tế cho khía cạnh này trong tập Test.")

# Vẽ và lưu Ma trận nhầm lẫn (Confusion Matrices)
# plot_confusion_matrices(true_sentiment, pred_sentiment, true_aspects, pred_aspects)

Đang đánh giá mô hình BiLSTM trên tập Test...

=== BÁO CÁO TỔNG THỂ (OVERALL SENTIMENT) ===
              precision    recall  f1-score   support

         neg       0.96      0.90      0.93      1048
         neu       0.59      0.78      0.67       324
         pos       0.93      0.87      0.90       633

    accuracy                           0.87      2005
   macro avg       0.83      0.85      0.83      2005
weighted avg       0.89      0.87      0.88      2005


=== Báo Cáo Mean 6 ASPECTS
              precision    recall  f1-score   support

         neg       0.92      0.82      0.87       805
         neu       0.77      0.68      0.72       410
         pos       0.94      0.89      0.92      1291

   micro avg       0.90      0.84      0.87      2506
   macro avg       0.87      0.80      0.83      2506
weighted avg       0.90      0.84      0.87      2506


=== BÁO CÁO CHI TIẾT TỪNG KHÍA CẠNH ===

--- Khía cạnh: NỘI DUNG (n = 617) ---
              precision    recall  f1-